In [1]:
# =========================================================
# DAILY ERA5 HEAT INDEX EXTRACTION
# 11 AM – 5 PM | APRIL 2026
# DISTRICT-WISE DAILY DOWNLOAD
# =========================================================

# =========================================================
# 1. INSTALL PACKAGES
# =========================================================

# Run once in VSCode terminal:
# pip install earthengine-api geemap geopandas pandas numpy openpyxl


# =========================================================
# 2. IMPORTS
# =========================================================

import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
from datetime import datetime, timedelta


# =========================================================
# 3. INITIALIZE EARTH ENGINE
# =========================================================

ee.Authenticate()
ee.Initialize(project='areca-farm')


# =========================================================
# 4. LOAD GEOJSON
# =========================================================

geojson_path = "../../assets/district.geojson"

gdf = gpd.read_file(geojson_path)

print("GeoJSON Loaded")


# =========================================================
# 5. REMOVE COMPLEX BOUNDARIES
# =========================================================
# Helps avoid incomplete edge pixels
# and speeds up computation
# =========================================================

gdf["geometry"] = gdf.geometry.simplify(0.01)

print("Geometry Simplified")


# =========================================================
# 6. CONVERT TO EARTH ENGINE
# =========================================================

districts = geemap.geopandas_to_ee(gdf)


# =========================================================
# 7. DATE RANGE
# =========================================================

start_date = datetime(2021, 1, 1)
end_date   = datetime(2025, 1, 1)


# =========================================================
# 8. ERA5 COLLECTION
# =========================================================

era5 = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")


# =========================================================
# 9. HEAT INDEX FUNCTION
# =========================================================

def add_heat_index(img):

    # Temperature (°C)
    t = img.select('temperature_2m').subtract(273.15)

    # Dewpoint (°C)
    d = img.select('dewpoint_temperature_2m').subtract(273.15)

    # Relative Humidity
    rh = (
        d.expression(
            '''
            100 * (
                exp((17.625 * Td)/(243.04 + Td)) /
                exp((17.625 * T)/(243.04 + T))
            )
            ''',
            {
                'Td': d,
                'T': t
            }
        )
    )

    # Heat Index
    hi = (
        t.expression(
            '''
            -8.784695 +
            1.61139411*T +
            2.338549*RH -
            0.14611605*T*RH -
            0.012308094*(T**2) -
            0.016424828*(RH**2) +
            0.002211732*(T**2)*RH +
            0.00072546*T*(RH**2) -
            0.000003582*(T**2)*(RH**2)
            ''',
            {
                'T': t,
                'RH': rh
            }
        )
    ).rename('HI')

    return hi.copyProperties(img, ['system:time_start'])


# =========================================================
# 10. DAILY LOOP
# =========================================================

all_days = []

current = start_date

while current < end_date:

    next_day = current + timedelta(days=1)

    print(f"\nProcessing: {current.strftime('%Y-%m-%d')}")

    # -----------------------------------------------------
    # FILTER DAILY DATA
    # -----------------------------------------------------

    daily = (
        era5
        .filterDate(
            current.strftime('%Y-%m-%d'),
            next_day.strftime('%Y-%m-%d')
        )
        .filter(ee.Filter.calendarRange(11, 16, 'hour'))
        .map(add_heat_index)
    )

    # -----------------------------------------------------
    # DAILY 11AM–5PM MEAN
    # -----------------------------------------------------

    daily_hi = daily.mean()

    # -----------------------------------------------------
    # DISTRICT MEAN
    # -----------------------------------------------------

    stats = daily_hi.reduceRegions(
        collection=districts,
        reducer=ee.Reducer.mean(),
        scale=11132,
        tileScale=4
    )

    # -----------------------------------------------------
    # DOWNLOAD SMALL DAILY TABLE
    # -----------------------------------------------------

    features = stats.getInfo()['features']

    rows = []

    for f in features:

        props = f['properties']

        rows.append({
            'date': current.strftime('%Y-%m-%d'),
            'district': props.get('dtname'),
            'heat_index': props.get('mean')
        })

    df_day = pd.DataFrame(rows)

    print("Collected Successfully")

    all_days.append(df_day)

    current = next_day


# =========================================================
# 11. COMBINE ALL DAYS
# =========================================================

daily_df = pd.concat(all_days, ignore_index=True)

print("\nDaily Collection Complete")


# =========================================================
# 12. MONTHLY DISTRICT MEAN
# =========================================================

monthly_df = (
    daily_df
    .groupby('district', as_index=False)
    ['heat_index']
    .mean()
)

monthly_df.rename(
    columns={'heat_index': 'April_2026_HI'},
    inplace=True
)

print("\nMonthly Means Computed")


# =========================================================
# 13. MERGE BACK TO GEOJSON ATTRIBUTES
# =========================================================

final_df = gdf.merge(
    monthly_df,
    left_on='dtname',
    right_on='district',
    how='left'
)

# Remove geometry column
final_df = final_df.drop(columns='geometry')

print("\nFinal CSV Ready")


# =========================================================
# 14. EXPORT CSV
# =========================================================

output_csv = "../../district_heat_index_april_2026.csv"

final_df.to_csv(output_csv, index=False)

print(f"\nCSV Saved:\n{output_csv}")

GeoJSON Loaded
Geometry Simplified

Processing: 2021-01-01
Collected Successfully

Processing: 2021-01-02
Collected Successfully

Processing: 2021-01-03
Collected Successfully

Processing: 2021-01-04
Collected Successfully

Processing: 2021-01-05
Collected Successfully

Processing: 2021-01-06
Collected Successfully

Processing: 2021-01-07
Collected Successfully

Processing: 2021-01-08
Collected Successfully

Processing: 2021-01-09
Collected Successfully

Processing: 2021-01-10
Collected Successfully

Processing: 2021-01-11
Collected Successfully

Processing: 2021-01-12
Collected Successfully

Processing: 2021-01-13
Collected Successfully

Processing: 2021-01-14
Collected Successfully

Processing: 2021-01-15
Collected Successfully

Processing: 2021-01-16
Collected Successfully

Processing: 2021-01-17
Collected Successfully

Processing: 2021-01-18
Collected Successfully

Processing: 2021-01-19
Collected Successfully

Processing: 2021-01-20
Collected Successfully

Processing: 2021-01-21
C

In [2]:
print(len(all_days))

365


In [14]:
# =========================================================
# PIXEL-WISE HEAT INDEX THRESHOLDS
# ODISHA | ERA5-LAND HOURLY
#
# BASELINE:
# 1990-2023
#
# HOURS:
# 06-12 UTC
# (~11:30 AM - 5:30 PM IST)
#
# METHOD:
# For each pixel:
#   Tmax hour within 06-12 UTC
#   Dewpoint from same Tmax hour
#   RH from Tmax/Td
#   Heat Index
#
# SEASONS:
#   JFM  = Jan-Mar
#   AMJ  = Apr-Jun
#   JJA  = Jun-Aug
#   ASO  = Aug-Oct
#   OND  = Oct-Dec
#
# PERCENTILES:
#   P80
#   P88
#   P95
#   P99
#
# OUTPUT:
#   20 GeoTIFFs
#
# =========================================================

import ee
import geemap
import geopandas as gpd

# =========================================================
# INITIALIZE EARTH ENGINE
# =========================================================

ee.Authenticate()
ee.Initialize(project="centering-sweep-480415-c5")

# =========================================================
# LOAD ODISHA GEOJSON
# =========================================================

geojson_path = "../../assets/district.geojson"

gdf = gpd.read_file(geojson_path)

gdf["geometry"] = gdf.geometry.simplify(
    0.05,
    preserve_topology=True
)

odisha = gdf.dissolve()

odisha_ee = geemap.geopandas_to_ee(odisha)

region = odisha_ee.geometry()

# Bounding box reduces payload size
export_region = region.bounds()

print("Odisha geometry loaded")

# =========================================================
# ERA5 LAND HOURLY
# =========================================================

era5 = (
    ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
    .filterDate("1990-01-01", "2024-01-01")
    .filter(ee.Filter.calendarRange(6, 12, "hour"))
)

print("ERA5 loaded")

# =========================================================
# ADD TEMPERATURE BAND
# =========================================================

def add_temp(img):

    T = (
        img.select("temperature_2m")
        .subtract(273.15)
        .rename("T")
    )

    return img.addBands(T)

era5 = era5.map(add_temp)

# =========================================================
# HEAT INDEX FROM Tmax HOUR
# =========================================================

def make_daily_hi(day):

    day = ee.Date(day)

    daily = era5.filterDate(
        day,
        day.advance(1, "day")
    )

    hottest = daily.qualityMosaic("T")

    T = hottest.select("T")

    Td = (
        hottest
        .select("dewpoint_temperature_2m")
        .subtract(273.15)
    )

    RH = Td.expression(
        """
        100 * (
            exp((17.625 * Td)/(243.04 + Td))
            /
            exp((17.625 * T)/(243.04 + T))
        )
        """,
        {
            "Td": Td,
            "T": T
        }
    )

    HI = T.expression(
        """
        -8.784695 +
        1.61139411*T +
        2.338549*RH -
        0.14611605*T*RH -
        0.012308094*T*T -
        0.016424828*RH*RH +
        0.002211732*T*T*RH +
        0.00072546*T*RH*RH -
        0.000003582*T*T*RH*RH
        """,
        {
            "T": T,
            "RH": RH
        }
    ).rename("HI")

    return HI.set(
        "system:time_start",
        day.millis()
    )

# =========================================================
# SEASON DEFINITIONS
# =========================================================

seasons = {
    "JFM": (1, 3),
    "AMJ": (4, 6),
    "JJA": (6, 8),
    "ASO": (8, 10),
    "OND": (10, 12)
}

percentiles = [80, 88, 95, 99]

# =========================================================
# BUILD SEASONAL DAILY COLLECTION
# =========================================================

def build_season_collection(start_month, end_month):

    season_ic = era5.filter(
        ee.Filter.calendarRange(
            start_month,
            end_month,
            "month"
        )
    )

    start = ee.Date("1990-01-01")
    end = ee.Date("2024-01-01")

    n_days = end.difference(start, "day")

    days = ee.List.sequence(
        0,
        n_days.subtract(1)
    )

    def process_day(offset):

        day = start.advance(offset, "day")

        month = ee.Number.parse(
            day.format("M")
        )

        valid = (
            month.gte(start_month)
            .And(month.lte(end_month))
        )

        img = ee.Image(
            ee.Algorithms.If(
                valid,
                make_daily_hi(day),
                None
            )
        )

        return img

    return ee.ImageCollection(
        days.map(process_day)
    )

# =========================================================
# SUBMIT ALL EXPORTS
# =========================================================

for season_name, months in seasons.items():

    print(f"Building {season_name}")

    daily_hi = build_season_collection(
        months[0],
        months[1]
    )

    for p in percentiles:

        print(
            f"Submitting {season_name} P{p}"
        )

        threshold = (
            daily_hi
            .reduce(
                ee.Reducer.percentile([p])
            )
            .rename(
                f"HI_P{p}"
            )
            .clip(region)
        )

        task = ee.batch.Export.image.toDrive(
            image=threshold,
            description=f"{season_name}_P{p}_1990_2023",
            folder="Heat_Index",
            fileNamePrefix=f"{season_name}_P{p}_1990_2023",
            region=export_region,
            scale=9000,
            maxPixels=1e13
        )

        task.start()

print("\n===================================")
print("ALL EXPORTS SUBMITTED")
print("===================================")

print("Expected outputs:")

for season_name in seasons:

    for p in percentiles:

        print(
            f"{season_name}_P{p}_1990_2023.tif"
        )

Odisha geometry loaded
ERA5 loaded
Building JFM
Submitting JFM P80
Submitting JFM P88
Submitting JFM P95
Submitting JFM P99
Building AMJ
Submitting AMJ P80
Submitting AMJ P88
Submitting AMJ P95
Submitting AMJ P99
Building JJA
Submitting JJA P80
Submitting JJA P88
Submitting JJA P95
Submitting JJA P99
Building ASO
Submitting ASO P80
Submitting ASO P88
Submitting ASO P95
Submitting ASO P99
Building OND
Submitting OND P80
Submitting OND P88
Submitting OND P95
Submitting OND P99

ALL EXPORTS SUBMITTED
Expected outputs:
JFM_P80_1990_2023.tif
JFM_P88_1990_2023.tif
JFM_P95_1990_2023.tif
JFM_P99_1990_2023.tif
AMJ_P80_1990_2023.tif
AMJ_P88_1990_2023.tif
AMJ_P95_1990_2023.tif
AMJ_P99_1990_2023.tif
JJA_P80_1990_2023.tif
JJA_P88_1990_2023.tif
JJA_P95_1990_2023.tif
JJA_P99_1990_2023.tif
ASO_P80_1990_2023.tif
ASO_P88_1990_2023.tif
ASO_P95_1990_2023.tif
ASO_P99_1990_2023.tif
OND_P80_1990_2023.tif
OND_P88_1990_2023.tif
OND_P95_1990_2023.tif
OND_P99_1990_2023.tif
